In [186]:
# Python 3.10.11
# %pip install -r requirements.txt > /dev/null
from args import *
from utils import *

In [187]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import random_split

from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import label_binarize

from torchmetrics import AUROC
from torch.utils.tensorboard import SummaryWriter


# 依赖导入
import pandas as pd


In [188]:
balance = True


# 导入label数据
label_df = pd.read_csv(sample_labels_file_path, index_col=0)

# 裁减样本数量，使得 0 - 1 样本数量一致
df_0 = label_df[label_df['label'] == 0]
df_1 = label_df[label_df['label'] == 1]
min_len = min(len(df_0), len(df_1))
new_df = pd.concat([df_0[:min_len], df_1[:min_len]], axis=0)

sample_key_list = label_df.index.to_list()
if balance:
    sample_key_list = new_df.index.to_list() # 均衡数据

print(f"label 样本数量: {len(sample_key_list)}")

# TODO(241225) 导入gene数据
# 根据 sample_key_list 为基准, 若模态数据中不存在 sample_key 则填充新数据
origin_gene_array = load_gene_data_by_sample_key(sample_key_list).values
origin_cnv_array = load_cnv_data_by_sample_key(sample_key_list).values

origin_wsi_array = load_wsi_data_by_sample_key(sample_key_list)
origin_report_array = load_report_data_by_sample_key(sample_key_list)

label 样本数量: 152


In [189]:
def get_mode(mode_name, column_len=1_000_000):

    global origin_gene_array, origin_cnv_array, origin_wsi_array, origin_report_array

    mode_list = []

    for key in mode_name:
        
        if key == "g":
            mode = np.copy(origin_gene_array)
        elif key == "c":
            mode = np.copy(origin_cnv_array)
        elif key == "w":
            mode = np.copy(origin_wsi_array)
        elif key == "r":
            mode = np.copy(origin_report_array)
        mode_list.append(mode[:, :column_len])
    return tuple(mode_list)


In [190]:
mode_name = "gcwr"
column_len = 64
gene_array, cnv_array, wsi_array, report_array = get_mode(mode_name, column_len)

In [191]:
print(column_len)

print("gene_array.shape, cnv_array.shape, wsi_array.shape, report_array.shape")
gene_array.shape, cnv_array.shape, wsi_array.shape, report_array.shape

64
gene_array.shape, cnv_array.shape, wsi_array.shape, report_array.shape


((152, 64), (152, 64), (152, 64), (152, 64))

In [192]:
label_array = label_df.values
if balance:
    label_array = new_df.values # 均衡数据

_, gene_dim = gene_array.shape
_, cnv_dim = cnv_array.shape
_, wsi_dim = wsi_array.shape
_, report_dim = report_array.shape
_, label_dim = label_array.shape

print(f"""
gene 数据维度:   {gene_dim}
cnv 数据维度:    {cnv_dim}
wsi 数据维度:    {wsi_dim}
report 数据维度: {report_dim}
""")

batch_size = 32

all_dataset = MultiOmicsDataset(gene_array, cnv_array, report_array, wsi_array, label_array)

# 假设 all_dataset 是一个 Dataset 对象
train_len = int(len(all_dataset) * 0.8)  # 80% 的数据用作训练集
test_len = len(all_dataset) - train_len  # 剩余的数据用作验证集

# 使用 random_split 分割数据集
train_val_dataset, test_dataset = random_split(all_dataset, [train_len, test_len])

train_loader = DataLoader(all_dataset, batch_size=batch_size, shuffle=True, num_workers=3, drop_last=False)
val_loader = DataLoader(train_val_dataset, batch_size=batch_size, shuffle=False, num_workers=3, drop_last=False)


gene 数据维度:   64
cnv 数据维度:    64
wsi 数据维度:    64
report 数据维度: 64



In [193]:
### 分割线

In [194]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# 构造含交叉注意力机制的Transformer
class TransformerEncoderLayerWithCrossAttention(nn.Module):
    def __init__(self, d_model, nhead, dropout=0.1, dim_feedforward=2048,):
        super(TransformerEncoderLayerWithCrossAttention, self).__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)

        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, src, src_mask=None):

                # 自注意力机制
        src2 = self.self_attn(src, src, src, attn_mask=src_mask)[0]
        src = src + self.dropout1(src2)
        src = self.norm1(src)

        # 将输入序列平均拆分为4等分
        seq_len = src.size(1)
        seg_len = seq_len // 4
        seg1 = src[:, :seg_len, :]
        seg2 = src[:, seg_len:2*seg_len, :]
        seg3 = src[:, 2*seg_len:3*seg_len, :]
        seg4 = src[:, 3*seg_len:, :]

        # 交叉注意力机制
        seg1_cross, _ = self.cross_attn(seg1, seg2, seg2)
        seg2_cross, _ = self.cross_attn(seg2, seg3, seg3)
        seg3_cross, _ = self.cross_attn(seg3, seg4, seg4)
        seg4_cross, _ = self.cross_attn(seg4, seg1, seg1)

        # 合并交叉注意力结果
        src_cross = torch.cat([seg1_cross, seg2_cross, seg3_cross, seg4_cross], dim=1)
        src = src + self.dropout2(src_cross)
        src = self.norm2(src)

        # 线性层和残差连接
        src2 = self.linear2(self.dropout(F.relu(self.linear1(src))))
        src = src + self.dropout3(src2)
        src = self.norm3(src)

        return src

In [195]:
class TransformerEncoderWithCrossAttention(nn.Module):
    def __init__(self, encoder_layer, num_layers, norm=None):
        super(TransformerEncoderWithCrossAttention, self).__init__()
        self.layers = nn.ModuleList([encoder_layer for _ in range(num_layers)])
        self.num_layers = num_layers
        self.norm = norm

    def forward(self, src, mask=None):
        output = src

        for layer in self.layers:
            output = layer(output, src_mask=mask)

        if self.norm is not None:
            output = self.norm(output)

        return output

# 定义模型结构
class MultiOmicsModel(nn.Module):
    def __init__(self, dropout_prob=0.2):
        super(MultiOmicsModel, self).__init__()

        # 分割线
        global gene_dim, cnv_dim, report_dim, wsi_dim, label_dim

        same_all_feature_dim = 8
        self.shared_hidden_gene_cnv = nn.Linear(gene_dim + cnv_dim, gene_dim)
        self.hidden_gene = nn.Sequential(nn.Linear(gene_dim, same_all_feature_dim))
        self.hidden_cnv = nn.Sequential(nn.Linear(gene_dim, same_all_feature_dim))

        self.fc_report = nn.Linear(report_dim, same_all_feature_dim)
        self.fc_wsi = nn.Linear(wsi_dim, same_all_feature_dim)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_prob)

        

        # 创建一个带有交叉注意力的Transformer编码器层
        # nn.MultiheadAttention(d_model, nhead, dropout=dropout) 
        d_model, nhead, dropout = same_all_feature_dim, 1, 0.1
        dim_feedforward = 8
        num_layers = 1
        encoder_layer = TransformerEncoderLayerWithCrossAttention(d_model, nhead, dropout, dim_feedforward)
        # 创建一个带有交叉注意力的Transformer编码器
        self.encoder = TransformerEncoderWithCrossAttention(encoder_layer, num_layers)

        self.flatten = nn.Flatten(start_dim=1)

        # 输出层
        self.lin = nn.Linear(same_all_feature_dim * 4, 2)


    def forward(self, gene_tensor, cnv_tensor, report_tensor, wsi_tensor):

        gene_cnv_feature = torch.concat([gene_tensor, cnv_tensor], dim=1)
        gene_cnv_feature = self.relu(self.shared_hidden_gene_cnv(gene_cnv_feature))
        gene_cnv_feature = self.dropout(gene_cnv_feature)

        report_feature = self.fc_report(report_tensor)
        report_feature = self.relu(report_feature)

        wsi_feature = self.fc_wsi(wsi_tensor)
        wsi_feature = self.relu(wsi_feature)

        gene_feature = self.hidden_gene(gene_cnv_feature)
        gene_feature = self.relu(gene_feature)

        cnv_feature = self.hidden_cnv(gene_cnv_feature)
        cnv_feature = self.relu(cnv_feature)

        all_feature = torch.stack([gene_feature, cnv_feature, report_feature, wsi_feature])
        all_feature = self.dropout(all_feature)

        # all_feature = all_feature.permute(1, 0, 2)
        # all_feature = self.flatten(all_feature)
        
        # out = self.lin(all_feature)
        # return out

        """
        start:  torch.Size([4, 32, 64])
        before enccode:  torch.Size([32, 4, 64])
        after enccode:  torch.Size([32, 4, 64])
        end:  torch.Size([4, 32, 64])
        """
        all_feature = all_feature.permute(1,0,2)
        all_feature = self.encoder(all_feature)

        all_feature = all_feature.permute(1, 0, 2)
        all_feature = self.relu(all_feature)

        all_feature = self.dropout(all_feature)

        all_feature = all_feature.permute(1, 0, 2)
        all_feature = self.flatten(all_feature)
        
        out = self.lin(all_feature)
        return out

In [196]:
print(f"""
gene 数据维度:   {gene_dim}
cnv 数据维度:    {cnv_dim}
wsi 数据维度:    {wsi_dim}
report 数据维度: {report_dim}
""")

# model(expr, cnv, report, sis)

# learning_rate = 0.001
# criterion = nn.BCEWithLogitsLoss()
# optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()  # 适用于二分类问题

# # tb001-0120-0826.tar.gz
# model = MultiOmicsModel(0.200).to(device) # 250120-0829
# optimizer = optim.Adam(model.parameters(), lr=0.00007705015) # 250120-0829

# # tb002-0120-0845.tar.gz
# model = MultiOmicsModel(0.2050).to(device) # 250120-0846
# optimizer = optim.Adam(model.parameters(), lr=0.00007205015) # 250120-0846

# # tb003-0120-0901.tar.gz
# model = MultiOmicsModel(0.2010).to(device) # 250120-0901
# optimizer = optim.Adam(model.parameters(), lr=0.00006805015) # 250120-0901

# tb004-0120-0927.tar.gz
model = MultiOmicsModel(0.20070).to(device) # 250120-0928
optimizer = optim.Adam(model.parameters(), lr=0.00007305015) # 250120-0928

# tb005-0120-0940.tar.gz
model = MultiOmicsModel(0.200780).to(device) # 250120-0940
optimizer = optim.Adam(model.parameters(), lr=0.00007305015) # 250120-0940

# tb006-0120-1006.tar.gz
model = MultiOmicsModel(0.200740).to(device) # 250120-1006
optimizer = optim.Adam(model.parameters(), lr=0.00007085015) # 250120-1006

# tb007-0120-1018.tar.gz
model = MultiOmicsModel(0.200700).to(device) # 250120-1018
optimizer = optim.Adam(model.parameters(), lr=0.00007035015) # 250120-1018

# tb008-0120-1025.tar.gz
model = MultiOmicsModel(0.200600).to(device) # 250120-1025
optimizer = optim.Adam(model.parameters(), lr=0.00006935015) # 250120-1025

# tb009-0120-1038.tar.gz
model = MultiOmicsModel(0.2005700).to(device) # 250120-1038
optimizer = optim.Adam(model.parameters(), lr=0.00006635015) # 250120-1038

# tb010-0120-1049.tar.gz
model = MultiOmicsModel(0.13005700).to(device) # 250120-1049
optimizer = optim.Adam(model.parameters(), lr=0.00036635015) # 250120-1049

# tb011-0120-1054.tar.gz
model = MultiOmicsModel(0.13005700).to(device) # 250120-1054
optimizer = optim.Adam(model.parameters(), lr=0.00066635015) # 250120-1054

# tb012-0120-1145.tar.gz
model = MultiOmicsModel(0.12005700).to(device) # 250120-1145
optimizer = optim.Adam(model.parameters(), lr=0.000069635015) # 250120-1145

# tb013-0120-1311.tar.gz
model = MultiOmicsModel(0.12005700).to(device) # 250120-1311
optimizer = optim.Adam(model.parameters(), lr=0.0002561635015) # 250120-1311

# tb014-0120-1319.tar.gz
model = MultiOmicsModel(0.12005700).to(device) # 250120-1319
optimizer = optim.Adam(model.parameters(), lr=0.0002061635015) # 250120-1319

# tb015-0120-1327.tar.gz
model = MultiOmicsModel(0.12005700).to(device) # 250120-1327
optimizer = optim.Adam(model.parameters(), lr=0.00021561635015) # 250120-1327

# tb015-0120-1327.tar.gz
model = MultiOmicsModel(0.00010700).to(device) # 250120-
optimizer = optim.Adam(model.parameters(), lr=0.00121561635015) # 250120-


gene 数据维度:   64
cnv 数据维度:    64
wsi 数据维度:    64
report 数据维度: 64



In [197]:
from thop import profile

# 假设 model 是你的 PyTorch 模型实例
# input_tensor 是一个代表输入数据的张量，其形状应与模型的输入层匹配
gene_tensor, cnv_tensor = torch.randn(1, gene_array.shape[1]), torch.randn(1, cnv_array.shape[1])
report_tensor, wsi_tensor = torch.rand(1, report_array.shape[1]), torch.randn(1, wsi_array.shape[1])

gene_tensor = gene_tensor.to(device)
cnv_tensor = cnv_tensor.to(device)
report_tensor = report_tensor.to(device)
wsi_tensor = wsi_tensor.to(device)

# 计算 FLOPs
flops, params = profile(model, inputs=(gene_tensor,cnv_tensor,report_tensor,wsi_tensor,))
print(f'FLOPs: {flops}')
print(f'Parameters: {params}')

[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
FLOPs: 11200.0
Parameters: 10594.0


In [198]:
assert 1 == 0

AssertionError: 

In [ ]:
from sklearn.metrics import roc_auc_score

def show_score(score, can_print=False, tb_index=None):
    auc = score["auc"]
    accuracy = score["accuracy"]
    f1_score = score["f1_score"]
    recall = score["recall"]
    specificity = score["specificity"]
    precision = score["precision"]
    mcc = score["mcc"]
    if can_print:
        print("AUC:", auc)
        print("Accuracy:", accuracy)
        print("F1 Score:", score["f1_score"])
        print("Recall:", f1_score)
        print("Specificity:", recall)
        print("Precision:", precision)
        print("MCC:", mcc)
    if tb_index is None:
        return
    global writer
    # 将指标写入 TensorBoard
    writer.add_scalar('Metrics/AUC', score["auc"], tb_index)
    writer.add_scalar('Metrics/Accuracy', score["accuracy"], tb_index)
    writer.add_scalar('Metrics/F1_Score', score["f1_score"], tb_index)
    writer.add_scalar('Metrics/Recall', score["recall"], tb_index)
    writer.add_scalar('Metrics/Specificity', score["specificity"], tb_index)
    writer.add_scalar('Metrics/Precision', score["precision"], tb_index)
    writer.add_scalar('Metrics/MCC', score["mcc"], tb_index)

In [ ]:
# 初始化TensorBoard SummaryWriter
writer = SummaryWriter(data_output_dir_path / f'runs/multi-model/{mode_name}')

num_epochs = 150
for epoch in range(num_epochs):

    correct = 0
    total = 0
    # 初始化变量来存储所有预测值和标签
    all_labels = []
    all_preds_prob = []

    model.train()
    # 导入 batch 数据
    for batch in train_loader:
        gene_tensor = batch["gene_tensor"].to(device)
        cnv_tensor = batch["cnv_tensor"].to(device)
        report_tensor = batch["report_tensor"].to(device)
        wsi_tensor = batch["wsi_tensor"].to(device)
        label_tensor = torch.squeeze(batch["label_tensor"]).to(torch.float32).to(device).view(-1, 1)
        label_tensor = label_tensor.squeeze()
        label_tensor = label_tensor.long()
        
        outputs = model(gene_tensor, cnv_tensor, report_tensor, wsi_tensor)

        optimizer.zero_grad()
        loss = criterion(outputs, label_tensor)
        loss.backward()
        optimizer.step()

        _, preds = torch.max(outputs, 1)  # 获取预测的类别
        total += label_tensor.size(0)
        correct += (preds == label_tensor).sum().item()

    print("training loss: ", loss.item())
    score = get_model_score(outputs, label_tensor.view(-1))
    show_score(score, can_print=False, tb_index=epoch)
    # writer.add_scalar('training/training_loss', loss.item(), epoch)

    # 计算准确率
    accuracy = correct / total
    print(f"training Accuracy: ({accuracy})")
    # writer.add_scalar('training/train_accuracy', accuracy, epoch)

    # 记录训练损失和准确率到同一个图表的不同系列
    writer.add_scalars('loss', {
        'train': loss.item(),
    }, epoch)

    writer.add_scalars('accuracy', {
        'train': accuracy,
    }, epoch)


    correct = 0
    total = 0
    # 初始化变量来存储所有预测值和标签
    all_labels = []
    all_preds_prob = []

    # 在每个epoch结束时进行验证
    model.eval()
    with torch.no_grad():
        for batch in val_loader:
            gene_tensor = batch["gene_tensor"].to(device)
            cnv_tensor = batch["cnv_tensor"].to(device)
            report_tensor = batch["report_tensor"].to(device)
            wsi_tensor = batch["wsi_tensor"].to(device)
            label_tensor = torch.squeeze(batch["label_tensor"]).to(torch.float32).to(device).view(-1, 1)
            label_tensor = label_tensor.squeeze()
            label_tensor = label_tensor.long()

            outputs = model(gene_tensor, cnv_tensor, report_tensor, wsi_tensor)

            loss = criterion(outputs, label_tensor)

            # outputs = model(gene_tensor)
            _, preds = torch.max(outputs, 1)  # 获取预测的类别
            total += label_tensor.size(0)
            correct += (preds == label_tensor).sum().item()
            
            # 假设outputs是模型的输出，形状为[24, 2]
            # 我们只关心正类的概率，所以取第二列（索引为1）
            all_preds_prob.extend(torch.sigmoid(outputs)[:, 1].cpu().numpy())
            all_labels.extend(label_tensor.cpu().numpy())

    writer.add_scalars('loss', {
        'valid': loss.item(),
    }, epoch)

    # 计算准确率
    accuracy = correct / total
    print(f"Validation Accuracy: ({accuracy})")
    # writer.add_scalar('validation/validation_accuracy', accuracy, epoch)

    # 计算AUC
    auc = roc_auc_score(all_labels, all_preds_prob)
    print(f'AUC: {auc}')
    # writer.add_scalar('validation/validation_auc', auc, epoch)

    writer.add_scalars('accuracy', {
        'valid': accuracy,
    }, epoch)

    writer.add_scalars('auc', {
        'valid': auc,
    }, epoch)

writer.close()

!scp -r /workspace/pyfaster/examples/model/data/output/runs adb6804e7b6df030ddab1a51a18e2ec7d2f9abad-woplfq@woplfq.ssh.ide.cloud.tencent.com:/tmp ;
!rm -rf /workspace/pyfaster/examples/model/data/output/runs


training loss:  0.8004433512687683
training Accuracy: (0.46710526315789475)
Validation Accuracy: (0.5619834710743802)
AUC: 0.5942622950819672
training loss:  0.727226734161377
training Accuracy: (0.5328947368421053)
Validation Accuracy: (0.6033057851239669)
AUC: 0.6076502732240436
training loss:  0.5769331455230713
training Accuracy: (0.5789473684210527)
Validation Accuracy: (0.628099173553719)
AUC: 0.6289617486338799
training loss:  0.5633809566497803
training Accuracy: (0.6513157894736842)
Validation Accuracy: (0.6694214876033058)
AUC: 0.6475409836065573
training loss:  0.6555879712104797
training Accuracy: (0.6578947368421053)
Validation Accuracy: (0.71900826446281)
AUC: 0.7040983606557377
training loss:  0.5925924181938171
training Accuracy: (0.7171052631578947)
Validation Accuracy: (0.743801652892562)
AUC: 0.7464480874316941
training loss:  0.6129662990570068
training Accuracy: (0.75)
Validation Accuracy: (0.7768595041322314)
AUC: 0.7960382513661203
training loss:  0.5472616553306

In [ ]:
# = get_mode(mode_name)
print(mode_name)
show_score(score, can_print=True)

gcwr
AUC: 1.0
Accuracy: 1.0
F1 Score: 1.0
Recall: 1.0
Specificity: 1.0
Precision: 1.0
MCC: 1.0


In [ ]:
# rrrr
# AUC: 0.6285714285714286
# Accuracy: 0.5833333333333334
# F1 Score: 0.5
# Recall: 0.5
# Specificity: 0.5
# Precision: 0.5
# MCC: 0.14285714285714285

In [ ]:
16_935_270
13_420_134
11_471_910

11471910